# 01 Run the checker on the sample evidence

Member 4. Runs `floodsitrep.nlp.checker.verify_claims` on the 20 label-derived samples in `evidence/samples/`.
For each tile it builds a few demo claims (one supported, one contradicted, some not verifiable) and prints the verdicts.
No GPU needed. Clear outputs before committing.

In [3]:
BRANCH = "main"

import os, subprocess, sys

if "google.colab" in sys.modules:
    if not os.path.exists("FloodVerify"):
        subprocess.run(["git", "clone", "https://github.com/aaarrvind/FloodVerify.git"], check=True)
    os.chdir("FloodVerify")
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull"], check=True)

REPO = os.getcwd() if os.path.isdir("src/floodsitrep") else os.path.abspath("../..")
sys.path.insert(0, os.path.join(REPO, "src"))
print("repo:", REPO)

repo: /content/FloodVerify/FloodVerify


In [4]:
import json
from pathlib import Path

import pandas as pd

from floodsitrep.nlp.checker import bin_of, class_fraction, verify_claims
from floodsitrep.nlp.schema import Claim

SAMPLES = Path("evidence/samples")
if not SAMPLES.exists():              # running from notebooks/m4/ on a laptop
    SAMPLES = Path("../../evidence/samples")

evidence = {p.stem: json.loads(p.read_text()) for p in sorted(SAMPLES.glob("*.json"))}
print(len(evidence), "sample evidence files")

20 sample evidence files


## Demo claims

`make_demo_claims` builds claims from the evidence itself: a true one, a wrong one (a different quantity word),
a claim about a cell that does not exist, and a claim about a cell with no usable imagery if the tile has one.

In [5]:
NEXT_BIN = {"none": "few", "few": "some", "some": "many", "many": "most", "most": "few"}


def make_demo_claims(ev):
    """Claims with known expected verdicts for one tile's evidence."""
    claims = []
    usable = [c for c in ev["cells"] if c["buildings"] > 0 and c.get("valid_fraction", 1.0) >= 0.5]
    if usable:
        c = usable[0]
        q = bin_of(class_fraction([c], "damaged"))
        wrong = NEXT_BIN[q]
        claims.append(Claim(text=f"{q} buildings in {c['cell']} are damaged", cell=c["cell"],
                            quantity=q, damage_class="damaged"))          # expect supported
        claims.append(Claim(text=f"{wrong} buildings in {c['cell']} are damaged", cell=c["cell"],
                            quantity=wrong, damage_class="damaged"))      # expect contradicted
    claims.append(Claim(text="Some buildings in r9c9 are destroyed", cell="r9c9",
                        quantity="some", damage_class="destroyed"))       # expect not_verifiable
    for c in ev["cells"]:
        if c.get("valid_fraction", 1.0) < 0.5:
            claims.append(Claim(text=f"Many buildings in {c['cell']} are destroyed", cell=c["cell"],
                                quantity="many", damage_class="destroyed"))  # expect not_verifiable
            break
    return claims

## Run the checker

In [6]:
def run_all(evidence):
    """Verdicts for the demo claims of every tile, as one table."""
    rows = []
    for tile_id, ev in evidence.items():
        for c in verify_claims(make_demo_claims(ev), ev):
            rows.append({"tile_id": tile_id, "claim": c.text, "cell": c.cell,
                         "quantity": c.quantity, "class": c.damage_class, "verdict": c.verdict})
    return pd.DataFrame(rows)


table = run_all(evidence)
table.head(12)

,tile_id,claim,cell,quantity,class,verdict
0,hurricane-florence_00000021_p1,most buildings in r0c1 are damaged,r0c1,most,damaged,supported
1,hurricane-florence_00000021_p1,few buildings in r0c1 are damaged,r0c1,few,damaged,contradicted
2,hurricane-florence_00000021_p1,Some buildings in r9c9 are destroyed,r9c9,some,destroyed,not_verifiable
3,hurricane-florence_00000034_p4,most buildings in r0c2 are damaged,r0c2,most,damaged,supported
4,hurricane-florence_00000034_p4,few buildings in r0c2 are damaged,r0c2,few,damaged,contradicted
5,hurricane-florence_00000034_p4,Some buildings in r9c9 are destroyed,r9c9,some,destroyed,not_verifiable
6,hurricane-florence_00000049_p2,most buildings in r0c1 are damaged,r0c1,most,damaged,supported
7,hurricane-florence_00000049_p2,few buildings in r0c1 are damaged,r0c1,few,damaged,contradicted
8,hurricane-florence_00000049_p2,Some buildings in r9c9 are destroyed,r9c9,some,destroyed,not_verifiable
9,hurricane-florence_00000072_p4,most buildings in r0c1 are damaged,r0c1,most,damaged,supported


In [7]:
print(table["verdict"].value_counts())
assert set(table["verdict"]) <= {"supported", "contradicted", "not_verifiable"}

verdict
not_verifiable    22
supported         20
contradicted      20
Name: count, dtype: int64


## Check by hand

Pick one tile and compare the verdicts with its evidence cells. Each verdict should be what you would say yourself.

In [12]:
tile = next(iter(evidence))
print(tile)
pd.DataFrame([{"cell": c["cell"], "buildings": c["buildings"], "valid": c.get("valid_fraction"),
               **c["class_fraction"]} for c in evidence[tile]["cells"]])

hurricane-florence_00000021_p1


,cell,buildings,valid,none,minor,major,destroyed
0,r0c0,0,1.0,0.0000,0.0000,0.0000,0.0
1,r0c1,3,1.0,0.3333,0.3333,0.3333,0.0
2,r0c2,0,1.0,0.0000,0.0000,0.0000,0.0
3,r1c0,1,1.0,1.0000,0.0000,0.0000,0.0
4,r1c1,5,1.0,0.2000,0.2000,0.2000,0.4
5,r1c2,0,1.0,0.0000,0.0000,0.0000,0.0
6,r2c0,0,1.0,0.0000,0.0000,0.0000,0.0
7,r2c1,1,1.0,1.0000,0.0000,0.0000,0.0
8,r2c2,0,1.0,0.0000,0.0000,0.0000,0.0


In [9]:
table[table["tile_id"] == tile]

,tile_id,claim,cell,quantity,class,verdict
0,hurricane-florence_00000021_p1,most buildings in r0c1 are damaged,r0c1,most,damaged,supported
1,hurricane-florence_00000021_p1,few buildings in r0c1 are damaged,r0c1,few,damaged,contradicted
2,hurricane-florence_00000021_p1,Some buildings in r9c9 are destroyed,r9c9,some,destroyed,not_verifiable
